Vehicle Data Preparation

Objectives:
- Inspect the vehicle dataset.
- Keep vehicles associated with the selected collisions.
- Assess data quality.
- Remove unnecessary columns.
- Export the prepared vehicle dataset.

Import Libraries

In [1]:
import pandas as pd
import numpy as np

Load Dataset

Load the cleaned collision dataset and the official vehicle dataset.

Only vehicles associated with collisions from 2010 onwards are retained to ensure consistency across the project.

In [2]:
df = pd.read_csv('/content/drive/MyDrive/UK-Accident-Prediction/dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv',
                 nrows=5)

In [3]:
df.head()

,collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,...,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding
0,2014214000919,2014,214000919,2,9,0,18,19,1,5,...,34,6,998,1,2,PEUGEOT 107,-1,E01010542,0,-1
1,2014554C16552,2014,554C16552,2,9,0,9,9,7,5,...,46,8,998,1,3,PEUGEOT 107,-1,E01032660,0,-1
2,201442I094703,2014,42I094703,4,9,0,3,3,7,3,...,58,9,998,1,5,PEUGEOT 107,-1,E01021412,0,-1
3,201443W158124,2014,43W158124,1,9,0,18,19,6,2,...,-1,-1,998,1,7,PEUGEOT 107,-1,-1,0,-1
4,201406L082426,2014,06L082426,2,9,0,18,19,8,4,...,20,4,998,1,1,PEUGEOT 107,-1,E01006253,0,-1


In [4]:
collision = set(
    pd.read_csv('/content/drive/MyDrive/UK-Accident-Prediction/collision_clean.csv',
     usecols=['collision_index'],
    dtype={'collision_index': 'string'})
    ['collision_index']
)

In [5]:
chunks = pd.read_csv(
    '/content/drive/MyDrive/UK-Accident-Prediction/dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv',
    chunksize=100_000,
    dtype={'collision_index': 'string', 'collision_ref_no': 'string'}
)

In [7]:
df = pd.concat(
    [
        chunk.loc[chunk['collision_index'].isin(collision)]
        for chunk in chunks
    ],
    ignore_index=True
)

/tmp/ipykernel_1563/3252651246.py:4: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks
/tmp/ipykernel_1563/3252651246.py:4: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks
/tmp/ipykernel_1563/3252651246.py:4: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks
/tmp/ipykernel_1563/3252651246.py:4: DtypeWarning: Columns (27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks
/tmp/ipykernel_1563/3252651246.py:4: DtypeWarning: Columns (27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks
/tmp/ipykernel_1563/3252651246.py:4: DtypeWarning: Columns (27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks
/tmp/ipykernel_1563/3252651246.

In [8]:
del collision

In [9]:
df.shape

(3400657, 32)

In [10]:
df.memory_usage(deep=True).sum() / 1024**2

np.float64(1488.1934385299683)

Exploration and Data Cleaning

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3400657 entries, 0 to 3400656
Data columns (total 32 columns):
 #   Column                                     Dtype 
---  ------                                     ----- 
 0   collision_index                            string
 1   collision_year                             int64 
 2   collision_ref_no                           string
 3   vehicle_reference                          int64 
 4   vehicle_type                               int64 
 5   towing_and_articulation                    int64 
 6   vehicle_manoeuvre_historic                 int64 
 7   vehicle_manoeuvre                          int64 
 8   vehicle_direction_from                     int64 
 9   vehicle_direction_to                       int64 
 10  vehicle_location_restricted_lane_historic  int64 
 11  vehicle_location_restricted_lane           int64 
 12  junction_location                          int64 
 13  skidding_and_overturning                   int64 
 14  hi

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
df.duplicated(subset=['collision_index', 'vehicle_reference']).sum()

np.int64(0)

In [14]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

,0


In [15]:
df.drop('collision_ref_no', axis=1, inplace=True)

In [16]:
columns = [
    'vehicle_manoeuvre_historic',
    'vehicle_manoeuvre',
    'vehicle_location_restricted_lane_historic',
    'vehicle_location_restricted_lane',
    'journey_purpose_of_driver_historic',
    'journey_purpose_of_driver'
]

In [18]:
for column in columns:
  print(df[column].value_counts(dropna=False).sort_index())

vehicle_manoeuvre_historic
-1      116254
 1       45414
 2      132257
 3      184040
 4      233481
 5      144763
 6       24667
 7      109549
 8       15964
 9      310568
 10      46904
 11      27385
 12      30381
 13      63096
 14      37539
 15      22080
 16     100675
 17     115812
 18    1540459
 99      99369
Name: count, dtype: int64
vehicle_manoeuvre
-1       13833
 1       46770
 2      137418
 3      188328
 4      239511
 5      150306
 6       25269
 7      112970
 8       16462
 9      319651
 10      48248
 11      28238
 12      31429
 13      64943
 14      38361
 15      22840
 19    1816253
 20        187
 99      99640
Name: count, dtype: int64
vehicle_location_restricted_lane_historic
-1      114162
 0     3102132
 1        1742
 2       14642
 3        1358
 4       11899
 5        4595
 6        9060
 7        1861
 8        3936
 9       37927
 99      97343
Name: count, dtype: int64
vehicle_location_restricted_lane
-1       12310
 0     3200186
 1     

In [20]:
df.drop([
    'vehicle_manoeuvre_historic',
        'vehicle_location_restricted_lane_historic',
        'journey_purpose_of_driver_historic'
],inplace=True,axis=1)

In [21]:
df.shape

(3400657, 28)

In [23]:
df.head()

,collision_index,collision_year,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane,junction_location,...,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding
0,201022D004634,2010,1,2,0,19,1,6,0,0,...,16,4,50,1,3,APRILIA RS50,-1,E01032137,0,-1
1,2010550D46157,2010,1,2,0,14,3,7,0,1,...,16,4,50,1,0,APRILIA RS50,-1,E01020427,0,-1
2,201037B045951,2010,1,2,0,19,1,5,0,0,...,16,4,50,1,2,APRILIA RS50,-1,E01030039,0,-1
3,201032D217210,2010,1,3,0,4,1,5,0,1,...,16,4,-1,-1,-1,-1,-1,E01026128,0,-1
4,2010170S32290,2010,1,2,0,4,1,5,0,1,...,26,6,49,1,4,APRILIA SR 50,-1,E01012229,0,-1


In [25]:
df[['age_of_driver', 'age_of_vehicle', 'engine_capacity_cc']].describe()

,age_of_driver,age_of_vehicle,engine_capacity_cc
count,3.400657e+06,3.400657e+06,3.400657e+06
mean,3.512233e+01,5.587383e+00,1.383028e+03
std,2.047958e+01,5.980322e+00,1.626238e+03
min,-1.000000e+00,-1.000000e+00,-1.000000e+00
25%,2.300000e+01,-1.000000e+00,-1.000000e+00
50%,3.500000e+01,5.000000e+00,1.388000e+03
75%,4.900000e+01,1.000000e+01,1.896000e+03
max,1.020000e+02,1.230000e+02,9.999900e+04


In [26]:
columns = ['age_of_driver', 'age_of_vehicle', 'engine_capacity_cc']
df[columns] = df[columns].replace(-1,float('nan'))
df[columns].isna().sum()

,0
age_of_driver,424394
age_of_vehicle,858410
engine_capacity_cc,874994


In [27]:
for column in columns:
  print(df[column].nlargest(10))

2713877    102.0
147425     101.0
675775     101.0
1354610    101.0
1771391    101.0
2813551    101.0
2822705    101.0
204276     100.0
228936     100.0
291554     100.0
Name: age_of_driver, dtype: float64
2218454    123.0
1012208    122.0
3087332    122.0
321656     118.0
1925531    111.0
1656186    110.0
2228217    108.0
2635040    105.0
2301751    104.0
1052789    103.0
Name: age_of_vehicle, dtype: float64
2844043    99999.0
108283     91000.0
799831     91000.0
1004167    91000.0
1145344    91000.0
2370744    91000.0
3219619    91000.0
799875     89790.0
2652965    86370.0
2647548    83000.0
Name: engine_capacity_cc, dtype: float64


In [28]:
for column in ['age_of_vehicle', 'engine_capacity_cc']:
    print(f'\nHighest values: {column}')

    display(
        df.loc[
            df[column].nlargest(5).index,
            [
                'collision_year',
                'vehicle_type',
                'generic_make_model',
                'age_of_vehicle',
                'engine_capacity_cc'
            ]
        ]
    )


Highest values: age_of_vehicle


,collision_year,vehicle_type,generic_make_model,age_of_vehicle,engine_capacity_cc
2218454,2023,90,-1,123.0,554.0
1012208,2022,9,-1,122.0,499.0
3087332,2022,9,-1,122.0,499.0
321656,2022,9,-1,118.0,1141.0
1925531,2013,9,-1,111.0,1560.0



Highest values: engine_capacity_cc


,collision_year,vehicle_type,generic_make_model,age_of_vehicle,engine_capacity_cc
2844043,2018,21,-1,18.0,99999.0
108283,2010,11,SCANIA Model Missing,11.0,91000.0
799831,2010,11,SCANIA Model Missing,10.0,91000.0
1004167,2016,11,SCANIA Model Missing,15.0,91000.0
1145344,2010,11,SCANIA Model Missing,10.0,91000.0


In [33]:
df.isnull().sum()

,0
collision_index,0
collision_year,0
vehicle_reference,0
vehicle_type,0
towing_and_articulation,0
vehicle_manoeuvre,0
vehicle_direction_from,0
vehicle_direction_to,0
vehicle_location_restricted_lane,0
junction_location,0


## Data Quality Notes

- No duplicate rows or duplicate vehicle identifiers were found.
- Missing codes (-1) in driver age, vehicle age and engine capacity were converted to NaN.
- Unusually high vehicle ages and engine capacities were observed.
- These values were retained because no verified correction rule was available.

Save Clean Dataset

Export the cleaned vehicle dataset.

The resulting dataset will be combined with the collision and casualty datasets during the data integration stage.

In [29]:
df.to_csv('/content/drive/MyDrive/UK-Accident-Prediction/vehicle_clean.csv', index=False)

Final Dataset

The vehicle dataset was successfully prepared for the next stage of the project.

Summary of the work completed:

- Filtered the dataset to include only vehicles involved in collisions from 2010 onwards.
- Assessed the overall quality of the data.
- Verified that no duplicate records were present.
- Confirmed that no missing values required treatment.
- Removed redundant identifier variables.
- Removed deprecated historical variables while retaining the latest coding standards.
- Exported a clean dataset ready for data integration.

In [30]:
print('Vehicle Dataset Summary')
print('='*50)
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(f'Missing Values: {df.isnull().sum().sum()}')
print(f'Duplicate Rows: {df.duplicated().sum()}')

Vehicle Dataset Summary
Rows: 3400657
Columns: 28
Missing Values: 2157798
Duplicate Rows: 0


In [31]:
df.head()

,collision_index,collision_year,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane,junction_location,...,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding
0,201022D004634,2010,1,2,0,19,1,6,0,0,...,16.0,4,50.0,1,3.0,APRILIA RS50,-1,E01032137,0,-1
1,2010550D46157,2010,1,2,0,14,3,7,0,1,...,16.0,4,50.0,1,0.0,APRILIA RS50,-1,E01020427,0,-1
2,201037B045951,2010,1,2,0,19,1,5,0,0,...,16.0,4,50.0,1,2.0,APRILIA RS50,-1,E01030039,0,-1
3,201032D217210,2010,1,3,0,4,1,5,0,1,...,16.0,4,NaN,-1,NaN,-1,-1,E01026128,0,-1
4,2010170S32290,2010,1,2,0,4,1,5,0,1,...,26.0,6,49.0,1,4.0,APRILIA SR 50,-1,E01012229,0,-1
